Guias usadas:
- Creación, conexión con cliente Supabase y extracción de data https://supabase.com/docs/reference/python/initializing

## Previamente en SQL EDITOR - Supabase

**Consideraciones** para hacer en el proyecto de Supabase. En la sección de SQL Editor. <p>
- Creación de esquema correspondiente: `create schema raw;`

## Librerias

In [1]:
import sys
sys.executable

'c:\\Users\\Angelica\\Documents\\Temporal-Carrera\\PerceivoAI\\REPOS_GITHUB\\.venv\\Scripts\\python.exe'

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from tqdm import tqdm
from pathlib import Path
import numpy as np
import json
from sqlalchemy import create_engine
from sqlalchemy import inspect

## Funciones Extract

In [ ]:
def obtener_datos(endpoint, login, authtoken):
    # Asegurar que la carpeta exista
    # Path("data/raw").mkdir(parents=True, exist_ok=True)


    datos = []
    page = 1
    barra= tqdm(desc=f"Descargando '{endpoint}'", unit=" páginas")

    while (True):
        url = f"https://api.jumpseller.com/v1/{endpoint}.json?page={page}&limit=50"
        r = requests.get(url, auth=HTTPBasicAuth(login, authtoken))
        if r.status_code != 200:
            print(f"❌ Error al obtener {endpoint}:", r.status_code)
            break

        data = r.json()
        if not data:
            break

        datos.extend(data)
        barra.update(1) # avanza la barra en 1 unidad
        page += 1

    barra.close()


    print(f"✅ Data obtenida: {endpoint.title()}. Total páginas {page-1}\n")
    datos= pd.json_normalize(datos)
    return datos

# datos.to_parquet(f"data/raw/{endpoint}_raw.parquet", index=False) # se almacena la data raw
    # datos.to_csv(f"data/raw/{endpoint}_raw.csv", index=False)


Probar que si se carga un archivo a **supabase**
- Guia oficial(click en Connect y seleccionar el proyecto para obtener las credenciales): 
  - https://supabase.com/dashboard/project/mbzsgavapsezblkypfwi?showConnect=true
  - https://supabase.com/docs/guides/troubleshooting/using-sqlalchemy-with-supabase-FUqebT
  
- pip install python-dotenv sqlalchemy psycopg2

In [ ]:
# para evitar problema como: can't adapt type 'numpy.ndarray'
# serializamos los valores array a str
def to_serializable(obj):
    if isinstance(obj, np.ndarray):
        return [to_serializable(x) for x in obj.tolist()]
    elif isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_serializable(x) for x in obj]
    else:
        return obj  # valores simples


def preprocessing_supabase(data_extracted):
    
    col_to_serializar=[]
    for col in data_extracted.columns:
        if data_extracted[col].apply(lambda x: isinstance(x, (np.ndarray, dict, list))).any(): 
            col_to_serializar.append(col)
    # ['product.categories', 'product.images', 'product.variants', 'product.fields]

    # print(col_to_serializar)
    if col_to_serializar: # si hay columnas 
        for col in col_to_serializar:
            # data_extracted[col]=data_extracted[col].apply(lambda fila: json.dumps(fila.tolist()) if isinstance(fila, np.ndarray) else json.dumps(fila))

            data_extracted[col] = data_extracted[col].apply( lambda fila: json.dumps(to_serializable(fila)) )
            
        print(f'Dataframe serializado. \nColumnas Serializadas: {col_to_serializar}\n')
    else:
        print('Nada que serializar')
    # return data_extracted
        

In [ ]:
products_df= pd.read_parquet("data/raw/products_raw.parquet")
customers_df= pd.read_parquet("data/raw/customers_raw.parquet")
orders_df= pd.read_parquet("data/raw/orders_raw.parquet")
# data/raw/{endpoint}_raw.parquet

### Funciones Supabase
Conexion con el proyecto en supabase

In [ ]:
def conect2supabase():
    # Fetch variables
    USER = os.getenv("user")
    PASSWORD = os.getenv("password")
    HOST = os.getenv("host")
    PORT = os.getenv("port")
    DBNAME = os.getenv("dbname")

    # Construct the SQLAlchemy connection string
    DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

    # Create the SQLAlchemy engine
    engine = create_engine(DATABASE_URL)
    print('✅ Supabase: Conexión Exitosa')
    return engine
    # # Test the connection
    # try:
    #     with engine.connect() as connection:
    #         print("Connection successful!")
    # except Exception as e:
    #     print(f"Failed to connect: {e}")

In [54]:
def insert2supabase(engine, esquema, endpoint, df):
    table_name = f"{endpoint}_{esquema}"
    full_table_name = f"{esquema}.{table_name}"

    try:
        inspector = inspect(engine)

        # Verificar si la tabla ya existe en el esquema
        table_exists = inspector.has_table(table_name, schema=esquema)

        #☑️ SI no existe, Crear tabla desde cero
        if not table_exists:
            df.to_sql(
                table_name,
                con=engine,
                schema=esquema,
                if_exists="replace",  # replace crea la tabla si no existe
                index=False
            )
            print(f"💾 Tabla {full_table_name} creada con {len(df)} filas.")
            return

        #☑️ SI la tabla existe, se añade las nuevas filas

        # 1️⃣ Obtiene las columnas de la tabla en Supabase
        with engine.connect() as conn:
            query=f""" 
                SELECT column_name FROM information_schema.columns
                WHERE table_schema = '{esquema}'
                AND table_name = '{table_name}';
            """
            columnas_actuales= pd.read_sql(query, conn)['column_name'].tolist()

        # Filtrar df para no añadir columnas nuevas
        df= df[[c for c in df.columns if c in columnas_actuales]]


        # 2️⃣ Si la tabla ya existe, traer IDs existentes
        if esquema=='raw':
            id_name= f"{endpoint[:-1]}.id" # tengo que limpiar la ultima letra
        elif esquema=='clean':
            ID_MAP = {
                "products": "id_producto",
                "customers": "id_cliente",
                "orders": "id_orden",
                "orders_products": "id_orden"  
            }
            id_name=ID_MAP.get(endpoint)
    
        with engine.connect() as conn:
            query = f'SELECT "{id_name}" FROM {full_table_name}'
            existing_ids = pd.read_sql( query, conn )[id_name].tolist()

        # 3️⃣ Filtrar solo las filas nuevas
        df_new = df[~df[id_name].isin(existing_ids)]

        if not df_new.empty:
            df_new.to_sql(
                table_name,
                con=engine,
                schema=esquema,
                if_exists="append",
                index=False
            )
            print(f"💾 {len(df_new)} nuevas filas insertadas en {full_table_name}.")
        else:
            print(f"ℹ️ No hay nuevas filas para insertar en {full_table_name}.")

    except Exception as e:
        print(f"❌ Error insertando en {full_table_name}: {e}")


## Ejecución: Extract 
Jumpseller  → Supabase(data_raw)

Pasos:
1. Extraccion de API Jumpseller
2. Almacena en un diccionario cada data de cada endpoint
3. Se serializa las columnas necesarias
4. Se conecta con el cliente de supabase
5. Se inserta las tablas obtenidas a Supabase

In [11]:
load_dotenv()

login =  os.getenv('JUMPSELLER_LOGIN')
authtoken =  os.getenv('JUMPSELLER_AUTHTOKEN')

endpoints=[ "products","customers","orders"]

dataframes={}
for endpoint in endpoints:
    dataframes[f'df_{endpoint}']= obtener_datos(endpoint, login, authtoken)


Descargando 'products': 27 páginas [00:42,  1.59s/ páginas]


✅ Data obtenida: Products. Total páginas 27



Descargando 'customers': 38 páginas [00:23,  1.64 páginas/s]


✅ Data obtenida: Customers. Total páginas 38



Descargando 'orders': 47 páginas [00:48,  1.03s/ páginas]

✅ Data obtenida: Orders. Total páginas 47



In [12]:
dataframes.keys()

dict_keys(['df_products', 'df_customers', 'df_orders'])

In [13]:
for key, df in dataframes.items():
    print(key)
    preprocessing_supabase(df)

df_products
Dataframe serializado. 
Columnas Serializadas: ['product.categories', 'product.images', 'product.variants', 'product.fields']

df_customers
Dataframe serializado. 
Columnas Serializadas: ['customer.shipping_addresses', 'customer.billing_addresses', 'customer.customer_categories', 'customer.customer_additional_fields']

df_orders
Dataframe serializado. 
Columnas Serializadas: ['order.promotions', 'order.products', 'order.additional_fields', 'order.shipping_taxes']



In [14]:
engine_supabase= conect2supabase()

✅ Supabase: Conexión Exitosa


In [ ]:
esquema= 'raw'

for endpoint in tqdm(endpoints, desc='Insertando en Supabase'):
    df= dataframes[f'df_{endpoint}']
    insert2supabase(engine_supabase, esquema, endpoint, df)
        

Insertando en Supabase:  33%|███▎      | 1/3 [00:02<00:05,  2.93s/it]

💾 109 nuevas filas insertadas en raw.products_raw.


Insertando en Supabase:  67%|██████▋   | 2/3 [00:04<00:02,  2.02s/it]

ℹ️ No hay nuevas filas para insertar en raw.customers_raw.


Insertando en Supabase: 100%|██████████| 3/3 [00:06<00:00,  2.26s/it]

💾 552 nuevas filas insertadas en raw.orders_raw.


In [52]:
# insertando nuevamente
insert2supabase(engine_supabase,esquema, 'products', dataframes['df_products'])

ℹ️ No hay nuevas filas para insertar en raw.products_raw.


**Siguiente paso**:
- pruebas_transform: extraer los datos raw de Supabase y aplicar transformacion(limpieza, normalizacion, etc)